# Trajectories of magnetic soft robot

In [15]:
import numpy as np
import jax.numpy as jnp
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from softmobility import SoftBody, FlowBodySolver, shear_flow, oscillating_magnetic_field

np.set_printoptions(precision=3, linewidth=100, suppress=True, sign=" ")

## Parameters

In [16]:
yaml_data = """
# Variable Prefixes (Optional)
design_names:      
  - spring_k    
  - radius
  - length
  - moment
input_names:
  - magnetic
    
# Default Values (Optional)
defaults:
  spring_k: 10            
  refradius: 1
  radius: 0.25
  length0: 0.
  length1: 0.
  moment: 3

# Spheres (Mandatory)
spheres:
  - # Sphere 0 #################
    radius: refradius                    
    orientation: [0, 0, dof0]
    torque: 
      - "- moment * magnetic2 * sin(dof0)"
      - "+ moment * magnetic2 * cos(dof0)"
      - "- moment * magnetic1 * cos(dof0) + moment * magnetic0 * sin(dof0) - spring_k * dof0"              

  - # Sphere 1 #################
    radius: radius               
    position: [refradius + radius + length0, 0, 0]       
    # torque: [0, 0, spring_k * dof0]
    torque: [0, 0, spring_k * (dof0 + dof1)]

  - # Sphere 2 #################
    radius: radius               
    position: 
      - "refradius + radius + length0 + (2*radius + length1) * cos(dof1)"
      - "(2 * radius + length1) * sin(dof1)"
      - 0
    orientation: [0, 0, dof1]       
    torque: [0, 0, -spring_k * dof1]
"""

## Soft Plankton

In [17]:
mybody= SoftBody(yaml_data, verbose=True)

  Found variables: dof0, dof1, length0, length1, moment, radius, spring_k, magnetic0, magnetic1, magnetic2
  3D field inputs:  ['magnetic']
    Sphere 0
      Radius: 1
      Position: ['0', '0', '0']
      Orientation: ['0', '0', 'dof0']
      Coupling matrix C_H:
[['0', '0', '0'], ['0', '0', '0'], ['0', '0', '0'], ['0', '0', '-moment*sin(dof0)'], ['0', '0', 'moment*cos(dof0)'], ['moment*sin(dof0)', '-moment*cos(dof0)', '0']]
      Coupling matrix C_K:
[['0', '0'], ['0', '0'], ['0', '0'], ['0', '0'], ['0', '0'], ['-spring_k', '0']]
    Sphere 1
      Radius: radius
      Position: ['length0 + radius + 1', '0', '0']
      Orientation: ['0', '0', '0']
      Coupling matrix C_H:
[['0', '0', '0'], ['0', '0', '0'], ['0', '0', '0'], ['0', '0', '0'], ['0', '0', '0'], ['0', '0', '0']]
      Coupling matrix C_K:
[['0', '0'], ['0', '0'], ['0', '0'], ['0', '0'], ['0', '0'], ['spring_k', 'spring_k']]
    Sphere 2
      Radius: radius
      Position: ['length0 + radius + (length1 + 2*radius)*cos(d

In [18]:
tensors = mybody.compute_mobility_problem()
print("\nMobility Matrix M_H (multiplied by mu):")
print(tensors.M_H)
print("\nMobility Matrix M_K (multiplied by mu):")
print(tensors.M_K)


Mobility Matrix M_H (multiplied by mu):
[[ 0.     0.     0.   ]
 [ 0.     0.001  0.   ]
 [ 0.     0.     0.017]
 [ 0.     0.     0.   ]
 [ 0.     0.     0.085]
 [ 0.    -0.057  0.   ]
 [ 0.    -0.061  0.   ]
 [ 0.     0.106  0.   ]]

Mobility Matrix M_K (multiplied by mu):
[[  0.      0.   ]
 [ -0.116   0.106]
 [  0.      0.   ]
 [  0.      0.   ]
 [  0.      0.   ]
 [  0.966   2.6  ]
 [ -1.169  -2.247]
 [ -2.247 -11.638]]


In [19]:
# Create a shear flow with shear rate 1
no_flow = shear_flow(shear_rate=0)
# Create a magnetic field 
magnetic_field = oscillating_magnetic_field(amp_x=-1, amp_y=1, omega=1)

print(magnetic_field.vector(time=0))
print(magnetic_field.vector(time=0.2))

[-1.  0.  0.]
[-1.     0.199  0.   ]


In [20]:
# parameters
final_time = 128 * jnp.pi  
dt = 0.1

mybody.set_design_defaults(new_dict={'spring_k': 10, 'radius': 0.5, 'moment': 1})

solver = FlowBodySolver(
    soft_body=mybody, 
    flow=no_flow, 
    input_map={"magnetic": magnetic_field}, 
    dt=dt, 
    init_orientation=[0, 0, 0], 
    integrator="RK2")
trajectory = solver.simulate(final_time=final_time)

OLD default param values: ['length0', 'length1', 'moment', 'radius', 'spring_k'] [  0.     0.     3.     0.25  10.  ]
NEW default param values: ['length0', 'length1', 'moment', 'radius', 'spring_k'] [  0.    0.    1.    0.5  10. ]
Time: 0.000 / 402.124  Integrator RK2
Time: 10.000 / 402.124  Integrator RK2
Time: 20.000 / 402.124  Integrator RK2
Time: 30.000 / 402.124  Integrator RK2
Time: 40.000 / 402.124  Integrator RK2
Time: 50.000 / 402.124  Integrator RK2
Time: 60.000 / 402.124  Integrator RK2
Time: 70.000 / 402.124  Integrator RK2
Time: 80.000 / 402.124  Integrator RK2
Time: 90.000 / 402.124  Integrator RK2
Time: 100.000 / 402.124  Integrator RK2
Time: 110.000 / 402.124  Integrator RK2
Time: 120.000 / 402.124  Integrator RK2
Time: 130.000 / 402.124  Integrator RK2
Time: 140.000 / 402.124  Integrator RK2
Time: 150.000 / 402.124  Integrator RK2
Time: 160.000 / 402.124  Integrator RK2
Time: 170.000 / 402.124  Integrator RK2
Time: 180.000 / 402.124  Integrator RK2
Time: 190.000 / 402.

In [21]:
# Extract position, orientation and dofs
positions = jnp.array([t[0] for t in trajectory])
orientations = jnp.array([t[1] for t in trajectory])
dofs = jnp.array([t[2] for t in trajectory])

# Time vector
t = jnp.linspace(0, final_time, len(trajectory))

# Create a subplot figure with 3 rows and 1 column
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=("DOF", "Position (X, Y, Z)", "Orientation (X, Y, Z)"),
    shared_xaxes=True,  # Share the x-axis for all plots (time)
    vertical_spacing=0.1  # Adjust vertical spacing (reduce clutter)
)

# Plot DOFs in the first subplot
fig.add_trace(go.Scatter(x=t, y=dofs[:, 0], mode='lines', name='DOF 0'), row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=dofs[:, 1], mode='lines', name='DOF 1'), row=1, col=1)

# Plot position components (X, Y, Z) in the second subplot
fig.add_trace(go.Scatter(x=t, y=positions[:, 0], mode='lines', name='Position X'), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=positions[:, 1], mode='lines', name='Position Y'), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=positions[:, 2], mode='lines', name='Position Z'), row=2, col=1)

# Plot orientation components (X, Y, Z) in the third subplot
fig.add_trace(go.Scatter(x=t, y=orientations[:, 0], mode='lines', name='Orientation X'), row=3, col=1)
fig.add_trace(go.Scatter(x=t, y=orientations[:, 1], mode='lines', name='Orientation Y'), row=3, col=1)
fig.add_trace(go.Scatter(x=t, y=orientations[:, 2], mode='lines', name='Orientation Z'), row=3, col=1)

# Update layout for titles and labels
fig.update_layout(
    title="Trajectory Data Over Time",
    xaxis_title="Time (t)",
    # yaxis_title="Values (Position/Orientation Components)",
    template="plotly_white",  # Set the background to white
    showlegend=True,  # Show legend to distinguish between the traces
    height=800,  # Set figure height
    width=800,   # Set figure width
    plot_bgcolor='white',  # Set the plot background to white
    paper_bgcolor='white'  # Set the paper background to white
)

# Show the plot
fig.show()